# Session 5 — Classical Information Theory

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

This is the **spine** of the course. Information theory gives us entropy, the KL
divergence, mutual information, and transfer entropy — and the same quantity
(**relative entropy / KL**) will reappear in Sinkhorn (S10), in quantum Sinkhorn (S14),
and in the capstone. We build the toolbox and meet directed information flow.

## 0. What you'll be able to do

- Read **entropy** as average surprise.
- Use **KL divergence** to compare two distributions (and know why it is asymmetric).
- Compute **mutual information** — the shared information between two variables.
- Measure **transfer entropy** — directed information flow ("who drives whom").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.infotheory.classical import (
    shannon_entropy, kl_divergence, mutual_information, transfer_entropy,
)

np.random.seed(42)
viz.use_course_style()

## 1. Surprise & entropy

A rare event is *surprising*; a certain event is not. **Entropy**
$H(p) = -\sum_x p(x)\log_2 p(x)$ is the average surprise, in **bits**. A fair coin has
1 bit; a biased coin has less.

In [ ]:
bias = np.linspace(0.001, 0.999, 200)
entropy_curve = [shannon_entropy([b, 1 - b]) for b in bias]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(bias, entropy_curve, color=viz.SOURCE_COLOR)
ax.axvline(0.5, color=viz.TARGET_COLOR, linestyle="--", label="fair coin (1 bit)")
ax.set_xlabel("coin bias  P(heads)")
ax.set_ylabel("entropy  H  (bits)")
ax.set_title("Entropy is maximal for a fair coin", pad=12)
ax.legend()
plt.show()

**Read the figure.** Entropy peaks at 1 bit for the fair coin (maximum uncertainty) and
falls to 0 at the certain outcomes $p=0$ or $p=1$ (no surprise). Uncertainty *is* information.

## 2. Comparing distributions: KL divergence

How many **extra bits** do you pay by modelling data drawn from $p$ as if it came from $q$?
That is the **Kullback–Leibler divergence** $D(p\|q) = \sum_x p(x)\log\frac{p(x)}{q(x)} \ge 0$.
It is *not* symmetric.

In [ ]:
p = np.array([0.7, 0.2, 0.1])
q = np.array([0.4, 0.4, 0.2])
print(f"D(p || q) = {kl_divergence(p, q):.3f} bits")
print(f"D(q || p) = {kl_divergence(q, p):.3f} bits   (different!)")
print(f"D(p || p) = {kl_divergence(p, p):.3f} bits")

**Read the output.** $D(p\|q) \neq D(q\|p)$ — order matters — and $D(p\|p)=0$. KL is the
**parent quantity** of this course: entropic-regularised optimal transport (Sinkhorn, S10)
is a KL projection, and its quantum version (S14) replaces KL by the quantum relative entropy.

## 3. Mutual information

**Mutual information** $I(X;Y)$ is the information two variables *share*: it is the KL
divergence between the joint $p_{XY}$ and the product of marginals $p_X p_Y$. It is zero iff
$X$ and $Y$ are independent. Let's tune a 2-bit joint from independent to perfectly correlated.

In [ ]:
independent = np.outer([0.5, 0.5], [0.5, 0.5])
correlated = np.array([[0.5, 0.0], [0.0, 0.5]])

ts = np.linspace(0.0, 1.0, 50)
mi_curve = [mutual_information((1 - t) * independent + t * correlated) for t in ts]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ts, mi_curve, color=viz.SOURCE_COLOR)
ax.set_xlabel("coupling  (0 = independent, 1 = perfectly correlated)")
ax.set_ylabel("mutual information  I(X;Y)  (bits)")
ax.set_title("Mutual information measures shared structure", pad=12)
plt.show()

**Read the figure.** $I(X;Y)$ rises from 0 (independent — knowing $X$ tells you nothing
about $Y$) to 1 bit (perfectly correlated — knowing $X$ fixes $Y$).

## 4. Directed flow: transfer entropy

Mutual information is symmetric — it cannot say *who drives whom*. **Transfer entropy**
$\mathrm{TE}_{Y\to X} = I(X_{t+1}; Y_t \mid X_t)$ (Schreiber, 2000) is **directed**: it asks
how much $Y$'s past tells us about $X$'s future, beyond $X$'s own past. We build a
source and a target that *copies the source with a one-step lag*.

In [ ]:
rng = np.random.default_rng(0)
source = rng.integers(0, 2, size=4000)
target = np.empty_like(source)
target[0] = 0
target[1:] = source[:-1]            # target copies source, lag 1

te_fwd = transfer_entropy(source, target)   # source -> target
te_bwd = transfer_entropy(target, source)   # target -> source
print(f"TE(source -> target) = {te_fwd:.3f} bits")
print(f"TE(target -> source) = {te_bwd:.3f} bits")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.bar(["source → target", "target → source"], [te_fwd, te_bwd],
       color=[viz.FLOW_COLOR, viz.TARGET_COLOR], alpha=0.9)
ax.set_ylabel("transfer entropy (bits)")
ax.set_title("Transfer entropy detects the direction of influence", pad=12)
plt.show()

**Read the figure.** Even though the two sequences are strongly related, transfer entropy
is **large only in the true causal direction** (source → target ≈ 1 bit) and ≈ 0 backwards.
This directionality is why TE is used to probe information flow in coupled systems.

## 5. A word of caution

Two honesty notes, in keeping with this course's standard:

- **Estimation bias.** TE (and mutual information) estimated from finite samples are
  **biased** — short recordings systematically over- or under-estimate them. Always check
  against shuffled/surrogate data.
- **PID is not unique.** Partial information decomposition (Williams–Beer) promises to split
  information into *redundant*, *unique*, and *synergistic* parts — but there is **no agreed
  definition** of redundancy. Treat synergy/redundancy claims with care.

## 6. Dictionary update

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | Born rule $|\langle x|\psi\rangle|^2$ |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| Markov kernel | quantum channel (CPTP map) |
| independent variables | product state (no entanglement) |
| **mutual information $I(X;Y)$** | **quantum mutual information $I(A{:}B)$** |

## 7. Your turn

1. **Entropy of a die.** Compute $H$ for a fair 6-sided die and for a loaded one. How many bits?
2. **KL both ways.** Pick your own $p, q$ and confirm $D(p\|q)\neq D(q\|p)$; when are they closest?
3. **Noisy copy.** Make the target copy the source only 80% of the time (flip 20%). Does
   $\mathrm{TE}_{source\to target}$ drop? Add a surrogate (shuffle the source) and compare.

## Conclusions & references

- **Entropy** = average surprise; **KL** = extra bits, the parent quantity of the course.
- **Mutual information** = shared structure; **transfer entropy** = directed flow.
- Estimates are biased; PID is not uniquely defined — stay circumspect.
- **Next (S6 — Information geometry):** distributions as a curved space, and the bridge to optimal transport.

**References:** Cover & Thomas, *Elements of Information Theory*; T. Schreiber, "Measuring Information Transfer", PRL 85, 461 (2000); D. MacKay, *Information Theory, Inference, and Learning Algorithms*. Previous: `notebooks/s04_composite.ipynb`.